In [18]:
import pandas as pd
import threshold_simulator
from utils import *
from pathlib import Path

%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
sessions_file = Path(__name__).resolve().parents[1] / "data" / "Sessions3.csv"
sessions_df = pd.read_csv(sessions_file)

sessions_df = sessions_df.sort_values(by="startChargeTime")

for month in range(1, 2):
    test_df = sessions_df[
        (pd.to_datetime(sessions_df["connectTime"]).dt.year == 2023)
        & (pd.to_datetime(sessions_df["connectTime"]).dt.month == month)
    ]

    test_df = test_df[test_df["DurationHrs"] > 0.5]
    test_df = test_df[test_df["cumEnergy_Wh"] > 0]

    sim = threshold_simulator.ThresholdSimulator(
        test_df,
        verbose=False,
        monte_carlo=True
    )

    power_profiles, prices, hourly_prices = sim.simulate()
    session_results = get_session_results(
        test_df, power_profiles, prices, sim.TOU, sim.delta_t
    )
    session_results.to_csv(f"results/{month}-2023-threshold.csv")

    agg_power_profile = aggregate_power_profiles(test_df, power_profiles, sim.delta_t)
    charging_revenue, TOU_costs = get_profit(
        test_df, power_profiles, prices, sim.delta_t, sim.TOU
    )

    demand_charge_kwh = max(agg_power_profile)
    demand_charge_cents = sim.cost_dc * demand_charge_kwh

    print("------------------------------------------------------------")

    print("Month", month)
    print("Total Profit", charging_revenue - TOU_costs - demand_charge_cents)
    print("Charging Revenue", charging_revenue)
    print("TOU Costs", TOU_costs)
    print("Demand Charge Costs (cents)", demand_charge_cents)
    print("Peak Power", demand_charge_kwh)

Optimizing sessions:   0%|          | 0/202 [00:00<?, ?it/s]


ValueError: Invalid dimensions (0, 1).

In [20]:
sessions_df

,dcosId,userId,vehicle_model,vehicle_maxChgRate_W,siteId,stationId,connectTime,startChargeTime,Deadline,energyReq_Wh,...,sch_centsPerOverstayHr,Duration,DurationHrs,choice,regular,scheduled,cumEnergy_Wh,peakPower_W,power,lastUpdate
0,24,605,500e,6600,23,7,2020-11-05T10:30:16,2020-11-05T10:31:09,NaN,NaN,...,200.0,0 days 03:43:57,3.73249,REGULAR,1,0,3281.0,6335,"[{'power_W': Decimal('6259'), 'timestamp': Dec...",2020-11-05T14:15:06
1,26,486,Model 3,24000,23,3,2020-11-11T07:39:55,2020-11-11T07:39:59,NaN,NaN,...,200.0,0 days 06:50:07,6.83527,REGULAR,1,0,33458.0,7005,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2020-11-11T14:30:06
2,30,620,Volt,3600,25,12,2020-11-13T16:19:55,2020-11-13T16:20:06,2020-11-14T04:15:00,18400.0,...,300.0,0 days 20:40:02,20.66722,SCHEDULED,0,1,15216.0,3450,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2020-11-14T13:00:08
3,31,618,Bolt,7200,23,6,2020-11-14T23:47:06,2020-11-14T23:47:16,NaN,NaN,...,400.0,0 days 02:12:51,2.21416,REGULAR,1,0,14378.0,6889,"[{'power_W': Decimal('6889'), 'timestamp': Dec...",2020-11-15T02:00:07
4,32,623,B-Class Electric Drive,6000,23,9,2020-11-16T11:38:44,2020-11-16T11:42:22,NaN,NaN,...,NaN,0 days 03:12:45,3.21249,REGULAR,1,0,12484.0,6852,"[{'power_W': Decimal('6813'), 'timestamp': Dec...",2020-11-16T14:55:07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6100,8205,1178,Mach-E,13000,25,11,2024-06-10T08:44:16,2024-06-10T08:44:50,NaN,NaN,...,300.0,0 days 03:45:15,3.75416,REGULAR,1,0,23588.0,6700,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2024-06-10T12:30:05
6099,8204,1431,niro,6000,25,15,2024-06-10T08:43:12,2024-06-10T08:45:07,2024-06-10T17:00:00,51000.0,...,300.0,0 days 08:14:59,8.24972,SCHEDULED,0,1,40316.0,6637,"[{'power_W': Decimal('6531'), 'timestamp': Dec...",2024-06-10T17:00:06
6101,8207,1561,Model Y Dual Motor,11000,25,13,2024-06-10T09:10:53,2024-06-10T09:11:11,2024-06-10T17:00:00,59516.0,...,300.0,0 days 07:43:54,7.73166,SCHEDULED,0,1,47518.0,6513,"[{'power_W': Decimal('6463'), 'timestamp': Dec...",2024-06-10T16:55:05
6102,8208,1060,e-Golf,40000,25,16,2024-06-10T11:22:22,2024-06-10T11:24:05,NaN,NaN,...,300.0,0 days 01:31:00,1.51666,REGULAR,1,0,9532.0,6737,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2024-06-10T12:55:05


In [22]:
agg_power_profile_all_sch = aggregate_power_profiles(
    test_df, power_profiles, sim.delta_t
)

charging_revenue, TOU_costs = get_profit(test_df, power_profiles, prices, sim.delta_t, sim.TOU)
profit_all_sch = charging_revenue - TOU_costs

profit_all_sch - sim.cost_dc * max(agg_power_profile_all_sch), max(
    agg_power_profile_all_sch
)

(69937.07026654345, 31.24316500640789)